In [ ]:
# Celda 1: Importaciones
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error, r2_score, mean_absolute_error
import shap

sns.set_style("whitegrid")

import sys
import os
sys.path.append(os.path.abspath('..'))

In [ ]:
# Celda 2: Cargar datos y aplicar feature engineering
df = pd.read_csv('data/raw/train.csv')

# Aplicar feature engineering
from src.features import feature_engineering
df = feature_engineering(df)

# Transformar target
df['SalePrice'] = np.log1p(df['SalePrice'])
y = df['SalePrice']
X = df.drop(['SalePrice', 'Id'], axis=1, errors='ignore')

print(f"Dataset listo para modelado: {X.shape}")

In [ ]:
# Celda 3: Preprocesamiento
from src.preprocessing import create_preprocessor

numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

preprocessor = create_preprocessor(numeric_features, categorical_features)
X_processed = preprocessor.fit_transform(X)

print(f"Features después de preprocesamiento: {X_processed.shape[1]}")

In [ ]:
# Celda 4: Split de datos
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test: {X_test.shape[0]} muestras")

In [ ]:
# Celda 5: Entrenamiento del modelo (XGBoost)
import xgboost as xgb

# Modelo base (buenos hiperparámetros iniciales)
model = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='rmsle'
)

model.fit(X_train, y_train, 
          eval_set=[(X_test, y_test)],
          early_stopping_rounds=50,
          verbose=False)

print("✅ Modelo XGBoost entrenado")

In [ ]:
# Celda 6: Evaluación del modelo
y_pred = model.predict(X_test)

# Volver a escala original
y_test_orig = np.expm1(y_test)
y_pred_orig = np.expm1(y_pred)

rmsle = np.sqrt(mean_squared_log_error(y_test_orig, y_pred_orig))
mae = mean_absolute_error(y_test_orig, y_pred_orig)
r2 = r2_score(y_test, y_pred)

print("="*60)
print("📊 RESULTADOS DEL MODELO")
print("="*60)
print(f"RMSLE  : {rmsle:.5f}")
print(f"MAE    : ${mae:,.2f}")
print(f"R²     : {r2:.4f}")
print("="*60)

In [ ]:
# Celda 7: Feature Importance
importances = model.feature_importances_
feature_names = preprocessor.get_feature_names_out()

feat_imp = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(data=feat_imp.head(20), x='Importance', y='Feature')
plt.title('Top 20 Features más Importantes')
plt.show()

print("Top 10 features:")
print(feat_imp.head(10))

In [ ]:
# Celda 8: Análisis SHAP (Interpretabilidad)
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:500])  # Muestra para velocidad

shap.summary_plot(shap_values, X_test[:500], feature_names=feature_names, max_display=15)

In [ ]:
# Celda 9: Guardar modelo y preprocessor
import os
os.makedirs('models', exist_ok=True)

joblib.dump(model, 'models/best_model.pkl')
joblib.dump(preprocessor, 'models/preprocessor.pkl')

print("💾 Modelos guardados correctamente en /models/")